# Bring-your-own OCR: Nemotron OCR v2 + document-pii-redactor

`document-pii-redactor`'s built-in Tesseract OCR works well for PDFs and good-quality images. For **difficult or blurred scans**, a stronger OCR pays off — this notebook plugs in NVIDIA's [`nvidia/nemotron-ocr-v2`](https://huggingface.co/nvidia/nemotron-ocr-v2) via the `detect(words=, boxes=)` interface.

> **Requirements (Nemotron side):** Linux amd64, an NVIDIA GPU with the CUDA toolkit (`nvcc` on `PATH`), and Python 3.12 — the package compiles a C++ CUDA extension at install time. The PII-redactor side runs anywhere.


## Install

Install PyTorch **first** (matching your CUDA toolkit), then Nemotron OCR with `--no-build-isolation`, then the redactor:


In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128   # example: CUDA 12.8


In [ ]:
!git lfs install && git clone https://huggingface.co/nvidia/nemotron-ocr-v2
%pip install --no-build-isolation -v ./nemotron-ocr-v2/nemotron-ocr
%pip install "document-pii-redactor[visual]"


## 1 — Run Nemotron OCR at word level

`lang="en"` selects the English v2 build, which handles **word-level** regions — exactly what the redactor wants. Each prediction is a dict with `text`, `confidence`, and `left`/`upper`/`right`/`lower` box coordinates.


In [ ]:
from nemotron_ocr.inference.pipeline_v2 import NemotronOCRV2

IMAGE = "report.png"          # <- your difficult/blurred document image

ocr = NemotronOCRV2(lang="en")   # downloads the v2_english checkpoint on first run
predictions = ocr(IMAGE, merge_level="word")

print(f"{len(predictions)} words")
predictions[:3]


## 2 — Convert predictions to `words` + pixel `boxes`

`detect(words=, boxes=)` expects boxes in **original-image pixel coordinates** as `[x0, y0, x1, y1]`. The helper below also scales up normalized (0–1) coordinates, in case your Nemotron build emits those:


In [ ]:
from PIL import Image

def nemotron_to_words_boxes(predictions, image_size):
    """Nemotron OCR prediction dicts -> (words, pixel boxes) for detect().

    Nemotron's upper/lower are bottom-left-origin (y grows upward), so in
    image pixel coordinates 'upper' is the LARGER y — each pair is sorted
    into the x0<=x1, y0<=y1 order detect() expects."""
    W, H = image_size
    words, boxes = [], []
    for p in predictions:
        if not p.get("text"):
            continue
        l, u, r, b = p["left"], p["upper"], p["right"], p["lower"]
        if max(l, u, r, b) <= 1.5:         # normalized output -> scale to pixels
            l, u, r, b = l * W, u * H, r * W, b * H
        x0, x1 = sorted((l, r))
        y0, y1 = sorted((u, b))
        words.append(p["text"])
        boxes.append([round(x0), round(y0), round(x1), round(y1)])
    return words, boxes

image = Image.open(IMAGE)
words, boxes = nemotron_to_words_boxes(predictions, image.size)
list(zip(words[:5], boxes[:5]))


## 3 — Detect and transform

Passing `words=`/`boxes=` skips Tesseract entirely; your exact boxes come back on the detected entities. From here every transform works as usual:


In [ ]:
from document_pii_redactor import ImagePIIRedactor

redactor = ImagePIIRedactor("ekacare/document-pii-redactor")
entities = redactor.detect(IMAGE, words=words, boxes=boxes)

for e in entities[:10]:
    print(f"{e.kind:6} {e.category:25} {str(e.text)[:40]!r}")


In [ ]:
redactor.redact(IMAGE, entities, mode="blur")


In [ ]:
deid = redactor.deidentify(IMAGE, entities)
deid.image


---

Notes:

- The visual-entity detector (signatures, QR codes, face photos, …) still runs on the image itself — bringing your own OCR only replaces the *text* extraction step.
- Nemotron OCR v2 is governed by the [NVIDIA Open Model License](https://www.nvidia.com/en-us/agreements/enterprise-software/nvidia-open-model-license/); `document-pii-redactor`'s own licensing is unchanged.
- No GPU handy? The [quickstart notebook](quickstart.ipynb) shows the same flow with the built-in Tesseract OCR.
